In [ ]:
import numpy as np
import matplotlib.pyplot as plt

shape    = (800, 450)  # Nombre de píxels (Ny, Nz) a la pantalla.
y_range  = (-8.0, 8.0) # Rang en el pla equatorial del forat negre.
z_range  = (-4.5, 4.5) # Rang en la direcció de l'eix de rotació.
M        = 1           # Massa del forat negre.

# Posem el paràmetre afí lam = 0 per a tots els fotons.
lam, dlam = 0, 0.01

# Angle de 0 a 2pi, que serà útil per dibuixar cercles.
contour  = np.linspace(0, 2 * np.pi, 100)

In [ ]:
# Definim les coordenades (t, x, y, z) inicials de l'eixam. 
t_screen = 0.0
x_screen = 5.0
y_screen = np.linspace(*y_range, shape[0])[:, None]
z_screen = np.linspace(*z_range, shape[1])[None, :]

# Passem a coordenades esfèriques, per la mètrica de Schwarzschild.
t  = t_screen
r  = np.sqrt(x_screen**2 + y_screen**2 + z_screen**2)
th = np.arctan(np.sqrt(x_screen**2 + y_screen**2) / z_screen)
ph = np.arctan2(y_screen, x_screen)

# Posem primer les components espacials de la 4-velocitat, alineada amb l'eix x.
# Aquestes expressions són les components de - d/dx en coordenades esfèriques.
dr_dl  = - np.cos(th) * np.cos(ph)
dth_dl = + np.sin(th) * np.cos(ph) / r
dph_dl = + np.sin(ph) / (r * np.cos(th))

# Ara normalitzem per tal que dt / dlambda = 1 i la 4-velocitat sigui nul·la.
norm2  = dr_dl**2 / (1 - 2 * M / r) + r**2 * (dth_dl**2 + np.sin(th)**2 * dph_dl**2)
factor = np.sqrt(norm2 / (1 - 2*M / r))

# Definim un eixam de fotons sobre la pantalla, amb els que farem el ray tracing.
swarm = np.zeros(shape + (8,))

swarm[..., 0] = t
swarm[..., 1] = r
swarm[..., 2] = th
swarm[..., 3] = ph

swarm[..., 4] = 1
swarm[..., 5] = dr_dl  / factor
swarm[..., 6] = dth_dl / factor
swarm[..., 7] = dph_dl / factor

# Comprovem que la 4-velocitat és efectivament nul·la
np.allclose(- (1 - 2 * M / r) * swarm[..., 4]**2 + swarm[..., 5]**2 / (1 - 2 * M / r) + r**2 * (swarm[..., 6]**2 + np.sin(th)**2 * swarm[..., 7]**2), 0)

In [ ]:
# Càlcul dels símbols de Christoffel, que es retornen amb la forma (Ny, Nz).
# Important: cal tenir en compte que són simètrics en els dos índexs de baix.
def christoffel(swarm, indices):
    # No necessitem la 4-velocitat per als Christoffels, la podem llençar
    t, r, th, ph, _, _, _, _  = swarm.transpose(2, 0, 1)

    if indices == (1, 0, 0):  # Γ^r_tt
        gamma = M * (r - 2 * M) / r**3

    elif (indices == (0, 0, 1)) or (indices == (0, 1, 0)):  # Γ^t_tr
        gamma = M / (r * (r - 2 * M))

    elif indices == (1, 1, 1):  # Γ^r_rr
        gamma = -M / (r * (r - 2 * M))

    elif indices == (1, 2, 2):  # Γ^r_θθ
        gamma = -(r - 2 * M)

    elif indices == (1, 3, 3):  # Γ^r_φφ
        gamma = -(r - 2 * M) * np.sin(th)**2

    elif (indices == (2, 1, 2)) or (indices == (2, 2, 1)):  # Γ^θ_rθ
        gamma = 1 / r

    elif indices == (2, 3, 3):  # Γ^θ_φφ
        gamma = -np.sin(th) * np.cos(th)

    elif (indices == (3, 1, 3)) or (indices == (3, 3, 1)):  # Γ^φ_rφ
        gamma = 1 / r

    elif (indices == (3, 2, 3)) or (indices == (3, 3, 2)):  # Γ^φ_θφ
        gamma = np.cos(th) / np.sin(th)

    else:
        gamma = 0 * r

    return gamma


In [ ]:

# Tasca 1: Cal implementar aquí una funció que calculi la derivada de l'array
# swarm respecte al paràmetre afí a partir de l'equació geodèsica.

# Tasca 2: Cal implementar un integrador Runge-Kutta 4 que aprofiti la funció
# anterior per evolucionar l'array swarm un pas de paràmetre afí dlam.


In [ ]:
# Definim una figura per poder representar imatges
###############################################################################
fig, ax = plt.subplots(1, 1, figsize = (8, 8))
plt.rc('font', size = 16)

# Per exemple, representem un símbol de Christoffel sobre la pantalla.
G = christoffel(swarm, (3, 2, 3))

ax.set_xlabel('y')
ax.set_ylabel('z')
ax.plot(2 * M * np.cos(contour), 2 * M * np.sin(contour), '--k')
ax.text(-1.0, 2.5, 'r = 2M')
ax.imshow(G.T, cmap = 'hot', origin = 'lower', extent = y_range + z_range);